In [0]:
%run ../../config

In [0]:
%run ./nb_series

In [0]:
import requests
import time

In [0]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

In [0]:
# URL base da API
base_url = 'https://api.stlouisfed.org/fred/series/observations'
# Chave da API
api_key = dbutils.secrets.get(scope='api_fred', key="api_key")
# Diretório de destino dos arquivos
dest_directory = PATH_POUSO + '/vol-landing-api_fred/SERIES'
# Lista de controle
series_not_uploaded = []

In [0]:
logger.info(f'base_url: {base_url}')
logger.info(f'dest_directory: {dest_directory}')
logger.info('Series to be loaded:')
for serie in series_list:
    logger.info(serie.serie_id + ' -> ' + serie.serie_description)

In [0]:
try:
    url = f"{base_url}?api_key={api_key}&series_id={series_list[0].serie_id}&file_type=json"
    response = requests.get(url)
    if response.status_code != 200:
        raise Exception("Verificar status da API")
except:
    raise Exception("Verificar status da API")

In [0]:
for serie in series_list:
    logger.info('----------------------------------------------------------')
    logger.info(serie.serie_description)
    url = f"{base_url}?api_key={api_key}&series_id={serie.serie_id}&file_type=json"
    if serie.frequency:
        url += f'&frequency={serie.frequency}'
    if serie.aggregation_method:
        url += f'&aggregation_method={serie.aggregation_method}'
    try:
        response = requests.get(url)
        content_type = response.headers.get('content-type')
        logger.info(f"{response.status_code}:{content_type} <- {url}")
        if response.status_code == 200 and 'json' in content_type:
            dbutils.fs.put(f"{dest_directory}/{serie.table_name}/{serie.table_name}.json", response.text, True)
        else:
            logger.warning(f"Failed to upload series: {serie.serie_description}")
            series_not_uploaded.append(serie.serie_description)
    except Exception as e:
        logger.error(e)
        series_not_uploaded.append(serie.serie_description)
    time.sleep(3)

In [0]:
if len(series_not_uploaded) > 0:
    for serie in series_not_uploaded:
        logger.error(serie)
    raise ValueError('Check unloaded series')